# Shared intrinsic profiles: train once, extract many

In a time series, the spatial profile of a spectrum barely changes, but the trace moves by
fractions of a pixel. `fit_shared_profile` trains **one intrinsic profile** on a few
integrations and evaluates it through each integration's trace, so the detector-level profile
follows the motion without refitting. Each integration keeps its own spectrum.

This tutorial simulates a transit with a moving trace, trains a shared profile, saves and
reloads it, extracts every integration and measures the transit depth.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import ndtr
import transitspectroscopy as ts
from transitspectroscopy import fit_shared_profile, SharedProfile

## Simulate a time series

Forty integrations; the trace moves up and down by up to 0.6 pixels; integrations 15 to 25
are in transit with a 1% depth.

In [ ]:
rng = np.random.default_rng(42)
nints, nrows, ncolumns, read_noise = 40, 32, 120, 5.
depth = 0.01
in_transit = (np.arange(nints) >= 15) & (np.arange(nints) < 25)

t = np.arange(ncolumns) / (ncolumns - 1)
base_trace = nrows / 2 + 2. * (t - .5)
widths = 1.5 + .3 * np.sin(2 * np.pi * t)
spectrum = 20000. * (1 + .2 * t)
edges = np.arange(nrows + 1)[:, None] - .5

images, variances, traces = [], [], []
for i in range(nints):
    trace = base_trace + .6 * np.sin(1.3 * i)
    profile = np.diff(ndtr((edges - trace) / widths), axis=0)
    expected = profile * spectrum * (1 - depth * in_transit[i])
    images.append(rng.poisson(expected) + rng.normal(0, read_noise, expected.shape))
    variances.append(expected + read_noise**2)
    traces.append(trace)
images, variances, traces = map(np.array, (images, variances, traces))
print(images.shape, traces.shape)

## Train the profile

We train on four out-of-transit integrations with a GP profile and fixed hyperparameters.
`optimize_hyperparameters=True` would also fit the GP amplitude and length scale.

In [ ]:
model = fit_shared_profile(
    images, variances, traces,
    training_indices=[0, 8, 30, 38],
    aperture_radius=5, spacing=1.,
    profile_method='gp', gp_options={'n_inducing': 12, 'length_scale': 30.},
    optimize_hyperparameters=False,
)
print(model.profile_method, model.options)
print('training time: {:.1f} s'.format(model.diagnostics['seconds']))

## Save, reload and evaluate

The model is saved as a portable `.npz` file (no pickle). Its fingerprint identifies exactly
which profile was used for an extraction.

In [ ]:
model.save('intrinsic_profile.npz')
model = SharedProfile.load('intrinsic_profile.npz')
print('fingerprint:', model.fingerprint()[:16])

P_first, P_moved = model.evaluate(traces[0]), model.evaluate(traces[1])
fig, axes = plt.subplots(1, 2, figsize=(10, 3), sharey=True)
for ax, P, i in zip(axes, (P_first, P_moved), (0, 1)):
    ax.imshow(P, aspect='auto', origin='lower'); ax.set_title(f'integration {i}')
axes[0].set_ylabel('Row');

## Extract every integration

`model.extract` returns three rows: column, flux and **inverse variance** (conditional on
the profile).

In [ ]:
flux = np.array([model.extract(images[i], traces[i], variances[i])[1] for i in range(nints)])
white = flux.sum(axis=1)
white /= np.median(white[~in_transit])

measured = 1 - np.mean(white[in_transit]) / np.mean(white[~in_transit])
error = np.std(white[~in_transit]) * np.sqrt(1 / in_transit.sum() + 1 / (~in_transit).sum())
print(f'injected depth {depth*100:.2f}%, measured {measured*100:.3f} +/- {error*100:.3f}%')

plt.figure(figsize=(8, 3))
plt.plot(white, 'o')
plt.xlabel('Integration'); plt.ylabel('Relative white-light flux');

See {doc}`../guides/shared-profile` for the model, its uncertainty approximations and
parallel training with Ray.